In [ ]:
# ============================================================
# 04_feature_engineering.ipynb — PRÉPARATION ML
# Objectif : transformer les insights EDA en features exploitables
# ============================================================

import sys
sys.path.append('..')
from src.utils import *

# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 1 — CHARGEMENT")
print("="*60 + "\n")
# -------------------------------------------------------

base = pd.read_csv('../data/processed/logistique_base.csv')
print(f"✓ Table chargée : {base.shape}")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 2 — REGROUPEMENT GÉOGRAPHIQUE PAR RÉGION")
print("   EDA Phase 7 : le Nordeste concentre les retards")
print("   27 états → 5 régions = moins de bruit, plus de signal")
print("="*60 + "\n")
# -------------------------------------------------------

# Officiel : les 5 régions du Brésil (IBGE)
regions_bresil = {
    # Nord
    'AC':'Nord','AP':'Nord','AM':'Nord','PA':'Nord','RO':'Nord','RR':'Nord','TO':'Nord',
    # Nordeste
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    # Centre-Ouest
    'DF':'Centre-Ouest','GO':'Centre-Ouest','MT':'Centre-Ouest','MS':'Centre-Ouest',
    # Sudeste
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    # Sul
    'PR':'Sul','RS':'Sul','SC':'Sul',
}

base['customer_region'] = base['customer_state'].map(regions_bresil)
base['seller_region']   = base['seller_state'].map(regions_bresil)

print("✓ Régions mappées")
print(f"\n--- Taux de retard par région CLIENT ---")
taux_region = (base.groupby('customer_region')['est_en_retard']
               .agg(['mean','count'])
               .sort_values('mean', ascending=False))
taux_region['mean'] = (taux_region['mean']*100).round(2)
print(taux_region)

# Feature dérivée : livraison inter-régionale ?
base['meme_region'] = (base['customer_region'] == base['seller_region']).astype(int)
print(f"\n✓ meme_region créée — % même région : {base['meme_region'].mean()*100:.1f}%")

taux_meme_region = base.groupby('meme_region')['est_en_retard'].mean() * 100
print(f"\n--- Taux retard selon meme_region ---")
print(taux_meme_region.round(2))


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 3 — RÉSOLUTION MULTICOLINÉARITÉ")
print("   EDA : volume_cm3 / product_weight_g corrélées à 0.81")
print("   Solution : créer une densité, garder les deux features")
print("="*60 + "\n")
# -------------------------------------------------------

# Densité = poids / volume → indique un colis compact et lourd
# vs un colis volumineux mais léger (ex: meuble en carton)
base['densite'] = base['product_weight_g'] / base['volume_cm3'].replace(0, np.nan)
base['densite'] = base['densite'].fillna(base['densite'].median())

print(f"✓ Densité calculée — médiane : {base['densite'].median():.3f} g/cm³")
print(f"\nCorrélation densite / retard_jours : "
      f"{base[['densite','retard_jours']].corr().iloc[0,1]:.3f}")

# On garde volume_cm3 ET product_weight_g malgré la corrélation :
# Random Forest gère bien la multicolinéarité (contrairement à la régression
# linéaire), et la feature importance permettra de trancher a posteriori


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 4 — VARIABLES TEMPORELLES ENRICHIES")
print("   EDA Phase 6 : pic de retard en mars, creux en juin")
print("="*60 + "\n")
# -------------------------------------------------------

# Flag sur les mois à risque identifiés en EDA (mars, fév, nov)
mois_a_risque = [2, 3, 11]
base['mois_a_risque'] = base['mois_achat'].isin(mois_a_risque).astype(int)

# Trimestre (regroupement plus stable que le mois brut)
base['trimestre'] = ((base['mois_achat'] - 1) // 3) + 1

print(f"✓ mois_a_risque créé (fév/mars/nov) : "
      f"{base['mois_a_risque'].mean()*100:.1f}% des commandes")
print(f"✓ trimestre créé")

taux_trim = base.groupby('trimestre')['est_en_retard'].mean() * 100
print(f"\n--- Taux retard par trimestre ---")
print(taux_trim.round(2))


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 5 — ENCODING DES VARIABLES CATÉGORIELLES")
print("   Cours 1 P4 — One-Hot avec drop_first=True")
print("="*60 + "\n")
# -------------------------------------------------------

# On encode les RÉGIONS (5 modalités), pas les 27 états
# → moins de dimensions, signal plus dense (confirmé en EDA)
base_encoded = pd.get_dummies(
    base,
    columns=['customer_region', 'seller_region'],
    prefix=['creg', 'sreg'],
    drop_first=True
)

print(f"✓ Colonnes avant encoding : {base.shape[1]}")
print(f"✓ Colonnes après encoding : {base_encoded.shape[1]}")
nouvelles_cols = [c for c in base_encoded.columns if c.startswith(('creg_','sreg_'))]
print(f"✓ Colonnes région créées : {nouvelles_cols}")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 6 — SÉLECTION FINALE DES FEATURES")
print("   Exclusion : identifiants, dates brutes, fuites")
print("="*60 + "\n")
# -------------------------------------------------------

# Colonnes à exclure du vecteur ML
cols_exclure = [
    'order_id', 'seller_id', 'product_id', 'customer_id',   # identifiants
    'customer_state', 'seller_state',                        # remplacées par région
    'cust_lat', 'cust_lng', 'seller_lat', 'seller_lng',       # remplacées par distance_km
    'retard_jours',                                           # variable cible continue (garder pour analyse, pas feature)
    'est_en_retard',                                          # TARGET
    'delai_livraison_total',                                  # ⚠️ FUITE : connu seulement après livraison
]

feature_cols = [c for c in base_encoded.columns if c not in cols_exclure]

X = base_encoded[feature_cols].copy()
y = base_encoded['est_en_retard'].copy()

print(f"⚠️  Exclusion de 'delai_livraison_total' : cette variable n'est connue")
print(f"    qu'APRÈS la livraison — l'utiliser serait du data leakage temporel")
print(f"    (même logique que review_score exclu du modèle churn)")

print(f"\n✓ Features finales ({len(feature_cols)}) :")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:>2}. {col}")

print(f"\n✓ Shape X : {X.shape}")
print(f"✓ Shape y : {y.shape}")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 7 — VÉRIFICATION NULLS RÉSIDUELS")
print("="*60 + "\n")
# -------------------------------------------------------

nulls = X.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) > 0:
    print("Nulls détectés :")
    print(nulls)
    for col in nulls.index:
        if X[col].dtype in ['float64','int64']:
            X[col] = X[col].fillna(X[col].median())
            print(f"  ✓ {col} imputé (médiane)")
else:
    print("✓ Aucun null — dataset propre")

print(f"\n✓ Vérification finale NaN : {X.isnull().sum().sum()}")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 8 — EXPORT")
print("="*60 + "\n")
# -------------------------------------------------------

import os
os.makedirs('../data/processed', exist_ok=True)

# Table complète (avec identifiants, pour traçabilité)
base_encoded.to_csv('../data/processed/logistique_features_full.csv', index=False)

# Table ML pure (X + y uniquement)
ml_ready = X.copy()
ml_ready['est_en_retard'] = y
ml_ready.to_csv('../data/processed/logistique_ml_ready.csv', index=False)

print(f"✓ logistique_features_full.csv : {base_encoded.shape}")
print(f"✓ logistique_ml_ready.csv      : {ml_ready.shape}")
print(f"\n→ Prochain : 05_machine_learning.ipynb")
print(f"  Modèle : Random Forest (gère bien la multicolinéarité")
print(f"  et les interactions non-linéaires identifiées en EDA)")


   ÉTAPE 1 — CHARGEMENT

✓ Table chargée : (110181, 21)

   ÉTAPE 2 — REGROUPEMENT GÉOGRAPHIQUE PAR RÉGION
   EDA Phase 7 : le Nordeste concentre les retards
   27 états → 5 régions = moins de bruit, plus de signal

✓ Régions mappées

--- Taux de retard par région CLIENT ---
                  mean  count
customer_region              
Nordeste         12.55  10085
Nord              8.73   2016
Centre-Ouest      6.47   6480
Sudeste           5.93  75722
Sul               5.74  15878

✓ meme_region créée — % même région : 62.4%

--- Taux retard selon meme_region ---
meme_region
0    7.50
1    6.04
Name: est_en_retard, dtype: float64

   ÉTAPE 3 — RÉSOLUTION MULTICOLINÉARITÉ
   EDA : volume_cm3 / product_weight_g corrélées à 0.81
   Solution : créer une densité, garder les deux features

✓ Densité calculée — médiane : 0.112 g/cm³

Corrélation densite / retard_jours : 0.008

   ÉTAPE 4 — VARIABLES TEMPORELLES ENRICHIES
   EDA Phase 6 : pic de retard en mars, creux en juin

✓ mois_a_risque 